# Run Matcher with TextMining cases and fullfill document

In [1]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.subcategory_factory import BenchmarkSubcategoryFactory
from benchmark.benchmark_suite import BenchmarkCaseManager, BenchmarkGoetterdammerung

test_suite_manager = BenchmarkCaseManager()
# BenchmarkSubcateroyFactory is optional and fullfill all tests rows for each product column in test_strategy.csv
test_suite_manager.load("test_strategy_generated.csv", BenchmarkSubcategoryFactory())


# A benchmark Group is a set of subgroups that defines a certain subcategory and contains associated tests encapsuled in a BenchmarkCase

# A BenchmarkCase holds a queryset which consists of one or multiple variety strings associated with a certain subcategory.
# every String is one run in API Client BenchmarkGoetterdammerung for Matcher API Götterdämmerung

benchmark_groups = test_suite_manager.get_benchmark_groups()

total = sum(
    len(case.get_queryset())
    for group in benchmark_groups.values()
    for sub_group in group.get_sub_groups().values()
    for case in sub_group.get_cases()
)
print("Total number of test cases: {}".format(total))

# four strategies exists: ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"] and "oui_match_strategy" see below.
match_strategies = ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
config = {"fuzzy_match_strategy_threshold": 0.8, "vector_match_strategy_threshold": 0.8}
target_system_b = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies, config=config)
count, count_all, ground_truth_suggestions, results = target_system_b.run(benchmark_groups)

for category in count.keys():
    print("{} - {} from {} cases succeeded".format(category, count[category], count_all[category]))

print("following suggestions are not matched:")
# these results failed in comparison with ground of truth value. they are listed as suggest
print(ground_truth_suggestions)

test_suite_manager.attach_results(results, count, count_all)

# details fulfill every row and column in the test section with suggestions if Factory is added in the constructor (see above).
# if results are attached to BenchmarkCaseManager saving them on following columns for every product
test_suite_manager.save_details("details_strategy.csv", with_results=False)
# saving statistics (runtime, passed, failed, count all)
test_suite_manager.save_results("results_strategy.csv")

Total number of test cases: 2274
Test - Lexical errors - 193 from 1106 cases succeeded
Test - Different Spelling - 162 from 718 cases succeeded
Test - Format errors - 122 from 450 cases succeeded
following suggestions are not matched:
{'SENTRON PAC3220', 'FPWIN Pro', 'Millenium MG', 'SENTRON PAC3200', 'OPC UA', 'Garage Door', 'Axeda Desktop', 'MELSEC Q Series Q20UDEHCPU', 'Smart Alarm', 'MELSOFT Update', 'Creo Elements/Direct', 'Data Acquisition', 'DNP Master', 'WinCC TIA', 'RUGGEDCOM RSG909R', 'OrionLX DNP', 'Visual BACnet', 'SYSMAC NJ/NX', 'Elevation C4', 'SCALANCE XRM334', 'SENTRON 7KT', 'ONS-S8 -', 'MELSOFT EM', 'ThingWorkx Industrial', 'Optigo Visual', 'SCALANCE XRH334', 'SIMATIC S7', 'Ethicon Endo-Surgery', 'SIMATIC S7-1500', 'ThingWorkx Kepware', 'Patient Portal', 'OMNTEC Proteus', 'TIA Portal', 'TBox LT2 All models', 'Logic Mobile', 'Digital Imaging', 'DNP Slave', 'Data Center', 'Plus EX', 'Power Generation', 'Linear eMerge', 'Orion5/Orion5r DNP', 'SENTRON PAC3200T', 'Switch Bo

# Conduct single benchmark cases with text miner and matcher

In [2]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.benchmark_suite import BenchmarkGoetterdammerung, BenchmarkCase


# Two querys for same product running against Götterdämmerung matcher API.
queryset = ["SIMATIC ET200", "SIMATIC ETT200"]
# Dummy Benchmark Case object: consider ground_truth when you modify queryset
benchmark_case = BenchmarkCase(test_type="Product", group=None, test_name="S7-1500", category="Test – Different Spelling", sub_category="different spelling - acronym", queryset=queryset, ground_truth=['contains("ET200")'], vendor="Siemens")

match_strategies = ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
target_system_a = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies)
# running against matcher API with Client method
results = target_system_a.run_single(case=benchmark_case)

for result in results:
    if result.is_match:
        print("succeeded:")
        print(result.suggest)
    else:
        print("failed:")
        print(result.suggest)

# 2nd test with other product
queryset = ["Siemens SIMATIC S7-1500 CPU PLC", "SIMATC S7-1500 CPU PLC", "6ES7518-4FX00-1AC0"]
benchmark_case = BenchmarkCase(test_type="Product", group=None, test_name="S7-1500", category="Test – Different Spelling", sub_category="different spelling - acronym", queryset=queryset, ground_truth=['contains("SIMATIC S7-1500")'], vendor="Siemens")

match_strategies = ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
config = {"fuzzy_match_strategy_threshold": 0.9  ,"vector_match_strategy_threshold": 0.9}
target_system_a = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies, config=config)
results = target_system_a.run_single(case=benchmark_case)

for result in results:
    if result.is_match:
        print("succeeded:")
        print(result.suggest)
    else:
        print("failed:")
        print(result.suggest)


succeeded:
SIMATIC ET200
succeeded:
SIMATIC ET200
succeeded:
SIMATIC S7-1500
succeeded:
SIMATIC S7-1500
failed:
None


# Conduct single query

In [3]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.benchmark_suite import BenchmarkGoetterdammerung, BenchmarkCase

queryset = ["CPU PLC Siemens und oder SIMATIC S7-1500 CPU PLC"]
benchmark_case = BenchmarkCase(test_type="Product", group=None, test_name="S7-1500", category="Test – Different Spelling", sub_category="different spelling - acronym", queryset=queryset, ground_truth=['contains("SIMATIC S7-1500")'], vendor="Siemens")

match_strategies = ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
config = {"fuzzy_match_strategy_threshold": 0.7  ,"vector_match_strategy_threshold": 0.7}
target_system_a = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies)
results = target_system_a.run_single(case=benchmark_case)

for result in results:
    if result.is_match:
        print("succeeded:")
        print(result.suggest)
        # if csaf documents are also matched showing them:
        # you can take a look in resources/CSAF ..
        print(result.csaf_ref)
    else:
        print("failed:")
        print(result.suggest)


succeeded:
SIMATIC S7-1500
[['CSAFPID-0001', 'ICSA-14-226-01'], ['CSAFPID-0001', 'ICSA-16-040-02'], ['CSAFPID-0003', 'ICSA-19-253-03'], ['CSAFPID-0001', 'ICSA-21-131-15'], ['CSAFPID-0002', 'ICSA-21-131-15'], ['CSAFPID-0079', 'ICSA-22-104-05'], ['CSAFPID-0004', 'ICSA-23-348-09'], ['CSAFPID-0005', 'ICSA-23-348-09'], ['CSAFPID-0007', 'ICSA-23-348-09'], ['CSAFPID-0008', 'ICSA-23-348-09'], ['CSAFPID-0028', 'ICSA-23-348-09'], ['CSAFPID-0029', 'ICSA-23-348-09'], ['CSAFPID-0031', 'ICSA-23-348-09'], ['CSAFPID-0032', 'ICSA-23-348-09'], ['CSAFPID-0042', 'ICSA-23-348-09'], ['CSAFPID-0056', 'ICSA-23-348-09'], ['CSAFPID-0001', 'ICSA-24-102-01'], ['CSAFPID-0023', 'ICSA-25-044-02'], ['CSAFPID-0024', 'ICSA-25-044-02'], ['CSAFPID-0031', 'ICSA-25-044-02'], ['CSAFPID-0032', 'ICSA-25-044-02'], ['CSAFPID-0037', 'ICSA-25-044-02'], ['CSAFPID-0038', 'ICSA-25-044-02'], ['CSAFPID-0039', 'ICSA-25-044-02'], ['CSAFPID-0040', 'ICSA-25-044-02'], ['CSAFPID-0041', 'ICSA-25-044-02'], ['CSAFPID-0044', 'ICSA-25-044-02'], 

# Good example with OUI

In [5]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.benchmark_suite import BenchmarkGoetterdammerung, BenchmarkCase


# Query contains OUI. Take a look in resources "latest_oui_lookup.json" for more examples
queryset = ["CPU PLC 38:4B:24 RF615R"]
benchmark_case = BenchmarkCase(test_type="Product", group=None, test_name="S7-1500", category="Test – Different Spelling", sub_category="different spelling - acronym", queryset=queryset, ground_truth=['contains("RF615R")'], vendor="Siemens")

match_strategies = ["oui_match_strategy", "exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
config = {"fuzzy_match_strategy_threshold": 0.8  ,"vector_match_strategy_threshold": 0.8}
target_system_a = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies)
results = target_system_a.run_single(case=benchmark_case)

for result in results:
    if result.is_match:
        print("succeeded:")
        print(result.suggest)
        if result.csaf_ref:
            print(result.csaf_ref)
    else:
        print("failed:")
        print(result.suggest)


succeeded:
SIMATIC RF615R
[['CSAFPID-0001', 'ICSA-19-192-04'], ['CSAFPID-0004', 'ICSA-21-159-13'], ['CSAFPID-0005', 'ICSA-21-159-13'], ['CSAFPID-0006', 'ICSA-21-159-13'], ['CSAFPID-0369', 'ICSA-22-167-14'], ['CSAFPID-0004', 'ICSA-24-256-07'], ['CSAFPID-0005', 'ICSA-24-256-07'], ['CSAFPID-0006', 'ICSA-24-256-07']]


# Conduct benchmark cases with text miner and matcher  with all threshold possibilities and matching strategy permutations

In [1]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.subcategory_factory import BenchmarkSubcategoryFactory
from benchmark.benchmark_suite import BenchmarkCaseManager, BenchmarkGoetterdammerung

test_suite_manager = BenchmarkCaseManager()
test_suite_manager.load("benchmark_selection.csv", BenchmarkSubcategoryFactory())
benchmark_groups = test_suite_manager.get_benchmark_groups()

total = sum(
    len(case.get_queryset())
    for group in benchmark_groups.values()
    for sub_group in group.get_sub_groups().values()
    for case in sub_group.get_cases()
)
print("Total number of test cases: {}".format(total))

# defines all permutations for three match strategies exact, fuzzy and vector
test_strategy_combination = [
    ["exact_match_strategy"],
    ["fuzzy_match_strategy"],
    ["vector_match_strategy"],

    ["exact_match_strategy", "fuzzy_match_strategy"],
    ["exact_match_strategy", "vector_match_strategy"],
    ["fuzzy_match_strategy", "vector_match_strategy"],
    ["vector_match_strategy", "fuzzy_match_strategy"],

    ['exact_match_strategy', 'fuzzy_match_strategy', 'vector_match_strategy'],
    ['exact_match_strategy', 'vector_match_strategy', 'fuzzy_match_strategy'],
    ['fuzzy_match_strategy', 'exact_match_strategy', 'vector_match_strategy'],
    ['fuzzy_match_strategy', 'vector_match_strategy', 'exact_match_strategy'],
    ['vector_match_strategy', 'exact_match_strategy', 'fuzzy_match_strategy'],
    ['vector_match_strategy', 'fuzzy_match_strategy', 'exact_match_strategy']
]

folder = "results"
from pathlib import Path
results_dir = Path(folder)
results_dir.mkdir(exist_ok=True)

for strategy_combination in test_strategy_combination:
    for i in range(1, 11):
        threshold = i / 10

        # exact matcher only once
        #if strategy_combination == ['exact_match_strategy'] and threshold != 1.0:
        #    continue

        strategie_name = "_".join([x.split("_")[0] for x in strategy_combination])


        #if os.path.exists(f"{results_dir}/{strategie_name}_{threshold}.csv"):
        #    continue

        ### uses for every match strategy combination
        config = {"fuzzy_match_strategy_threshold": threshold, "vector_match_strategy_threshold": threshold}
        target_system_b = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=strategy_combination, config=config)
        count, count_all, ground_truth_suggestions, results = target_system_b.run(benchmark_groups)

        for category in count.keys():
            print("{} - {} from {} cases succeeded".format(category, count[category], count_all[category]))

        print("following suggestions are not matched:")
        print(ground_truth_suggestions)

        test_suite_manager.attach_results(results, count, count_all)
        test_suite_manager.save_results(f"{folder}/system_2024_2026_{strategie_name}_{threshold}.csv")

Total number of test cases: 136
Test - Lexical errors - 16 from 67 cases succeeded
Test - Different Spelling - 8 from 42 cases succeeded
Test - Format errors - 4 from 27 cases succeeded
following suggestions are not matched:
{'SIMATIC S7'}
Test - Lexical errors - 16 from 67 cases succeeded
Test - Different Spelling - 8 from 42 cases succeeded
Test - Format errors - 4 from 27 cases succeeded
following suggestions are not matched:
{'SIMATIC S7'}
Test - Lexical errors - 16 from 67 cases succeeded
Test - Different Spelling - 8 from 42 cases succeeded
Test - Format errors - 4 from 27 cases succeeded
following suggestions are not matched:
{'SIMATIC S7'}
Test - Lexical errors - 16 from 67 cases succeeded
Test - Different Spelling - 8 from 42 cases succeeded
Test - Format errors - 4 from 27 cases succeeded
following suggestions are not matched:
{'SIMATIC S7'}
Test - Lexical errors - 16 from 67 cases succeeded
Test - Different Spelling - 8 from 42 cases succeeded
Test - Format errors - 4 from 2